## **#Importing necessary libraries for data manipulation**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib . pyplot as plt
import scipy.stats as stats

### **#Load tables into the notebook**

In [ ]:
orders = pd.read_csv("../content/drive/MyDrive/ADA2026_Instacart_Project/data/raw/instacart/orders.csv")
products = pd.read_csv("../content/drive/MyDrive/ADA2026_Instacart_Project/data/raw/instacart/products.csv")
aisles= pd.read_csv("../content/drive/MyDrive/ADA2026_Instacart_Project/data/raw/instacart/aisles.csv")
departments= pd.read_csv("../content/drive/MyDrive/ADA2026_Instacart_Project/data/raw/instacart/departments.csv")
orderprodprior= pd.read_csv("../content/drive/MyDrive/ADA2026_Instacart_Project/data/raw/instacart/order_products__prior.csv")
orderprodtrain= pd.read_csv("../content/drive/MyDrive/ADA2026_Instacart_Project/data/raw/instacart/order_products__train.csv")

**#Contingency Table**

In [ ]:
# Link items to departments and create a contingency table
merged_df = orderprodprior.merge(products[['product_id', 'department_id']], on='product_id')
task1_ct = pd.crosstab(merged_df['department_id'], merged_df['reordered'])
print(task1_ct)

reordered            0        1
department_id                  
1              1024542  1211890
2                21485    14806
3               437599   739188
4              3318581  6160710
5                66101    87595
6               169837    99416
7               932237  1757892
8                38964    58760
9               467046   399581
10               14623    19950
11              303539   143584
12              306489   402442
13             1225276   650301
14              311556   398013
15              579523   488535
16             1786795  3627221
17              441591   297075
18              178433   245369
19             1229577  1657973
20              412385   638864
21               41774    27371


**#Defining "Observed" and Computing Expected Frequencies**

In [ ]:
# 1. First, create the 'observed' contingency table
# Merge items with products to get department IDs, then cross-tabulate with reorders
merged_df = orderprodprior.merge(products[['product_id', 'department_id']], on='product_id')
observed = pd.crosstab(merged_df['department_id'], merged_df['reordered'])

# 2. Compute the necessary components for the Expected Frequency formula [1, 6]
row_totals = observed.sum(axis=1)       # Total items per department (21 rows)
col_totals = observed.sum(axis=0)       # Total First-time vs Reordered (2 columns)
grand_total = observed.sum().sum()      # Total population (n = 32,434,489) [7]

# 3. Apply the manual formula (Row Total * Col Total / Grand Total) [1, 2, 8]
# Use np.outer to multiply every row total by every column total efficiently
expected_array = np.outer(row_totals, col_totals) / grand_total

# 4. Convert to a DataFrame for better interpretability
expected_df = pd.DataFrame(expected_array, index=observed.index, columns=observed.columns)

# 5. Display the results
print("Observed Contingency Table (Actual Counts):")
print(observed)
print("\nExpected Frequency Table (Under H0):")
print(expected_df)

Observed Contingency Table (Actual Counts):
reordered            0        1
department_id                  
1              1024542  1211890
2                21485    14806
3               437599   739188
4              3318581  6160710
5                66101    87595
6               169837    99416
7               932237  1757892
8                38964    58760
9               467046   399581
10               14623    19950
11              303539   143584
12              306489   402442
13             1225276   650301
14              311556   398013
15              579523   488535
16             1786795  3627221
17              441591   297075
18              178433   245369
19             1229577  1657973
20              412385   638864
21               41774    27371

Expected Frequency Table (Under H0):
reordered                 0             1
department_id                            
1              9.176137e+05  1.318818e+06
2              1.489029e+04  2.140071e+04
3             

## **TASK 2**

# **#WEEKEND INDICATOR**

In [ ]:
# 1. Create the binary weekend indicator (1 for Weekend, 0 for Weekday)
# Logic: 0 is Sunday, 6 is Saturday. Everything else (1-5) is a Weekday.
orders['is_weekend_binary'] = orders['order_dow'].apply(lambda x: 1 if x in [0, 6] else 0)

# 2. Create an interpretable label column
orders['day_type'] = orders['is_weekend_binary'].map({1: 'Weekend', 0: 'Weekday'})

# 3. Display a summary frequency table to verify the distribution
summary_table = orders['day_type'].value_counts()
percentage_table = orders['day_type'].value_counts(normalize=True) * 100

print("Frequency Distribution of Day Type:")
print(summary_table)
print("\nPercentage Distribution:")
print(percentage_table)

Frequency Distribution of Day Type:
day_type
Weekday    2371417
Weekend    1049666
Name: count, dtype: int64

Percentage Distribution:
day_type
Weekday    69.317728
Weekend    30.682272
Name: proportion, dtype: float64


# **#Cross tabulate and Chi Square Test**

In [ ]:
# Cross-tabulation: Hour of Day vs Weekend Indicator
contingency_table = pd.crosstab(orders['order_hour_of_day'], orders['is_weekend_binary'])
contingency_table.columns = ['Weekday', 'Weekend']

print("Contingency Table: Hour of Day vs Weekend Indicator")
print("="*60)
print(contingency_table)

# Chi-Square Test of Independence
from scipy.stats import chi2_contingency

chi2, p_value, dof, expected_freq = chi2_contingency(contingency_table)

print("\n" + "="*60)
print("Chi-Square Test of Independence Results")
print("\n" + "="*60)
print(f"Chi-Square Statistic: {chi2:.4f}")
print(f"P-Value: {p_value:.4e}")
print(f"Degrees of Freedom: {dof}")
print(f"Significance Level: 0.05")
print(f"Result: {'Reject null hypothesis (variables are dependent)' if p_value < 0.05 else 'Fail to reject null hypothesis (variables are independent)'}")

Contingency Table: Hour of Day vs Weekend Indicator
                   Weekday  Weekend
order_hour_of_day                  
0                    15516     7242
1                     8081     4317
2                     4916     2623
3                     3648     1826
4                     3912     1615
5                     7265     2304
6                    23957     6572
7                    68139    23729
8                   127133    51068
9                   186175    71637
10                  204288    84130
11                  196699    88029
12                  184277    88564
13                  186586    91413
14                  189742    93300
15                  191592    92047
16                  187528    85025
17                  158644    70151
18                  129183    53729
19                   99569    41000
20                   72623    31669
21                   53185    24924
22                   41690    19778
23                   27069    12974

Chi-Square 

# **TASK 3**

In [ ]:
# 1. Calculate order size (total items) for every order
order_sizes = orderprodprior.groupby('order_id').size().reset_index(name='basket_size')

# 2. Define the categorisation function
def categorise_size(size):
    if size <= 5: return 'Small'
    elif size <= 15: return 'Medium'
    else: return 'Large'

# 3. Apply categories and merge reorder data
order_sizes['size_cat'] = order_sizes['basket_size'].apply(categorise_size)
merged_data = orderprodprior.merge(order_sizes[['order_id', 'size_cat']], on='order_id')

# 4. Create the 3x2 Contingency Table (Observed Frequencies)
# Rows: Small, Medium, Large | Columns: 0 (First-time), 1 (Reordered)
task3_ct = pd.crosstab(merged_data['size_cat'], merged_data['reordered'])

# 5. Perform Chi-Square Test
chi2, p, dof, expected = stats.chi2_contingency(task3_ct)

print("Task 3 Contingency Table (Observed):")
print(task3_ct)
print(f"\nCalculated Chi-Square: {chi2:.4f}")
print(f"P-Value: {p:.4f}")
print(f"Degrees of Freedom: {dof}")

Task 3 Contingency Table (Observed):
reordered        0        1
size_cat                   
Large      5639713  8218704
Medium     6466246  8928037
Small      1201994  1979795

Calculated Chi-Square: 20593.5645
P-Value: 0.0000
Degrees of Freedom: 2


# **#Contingency Table**

In [ ]:
# 1. Recalculate basket size for all orders
basket_sizes = orderprodprior.groupby('order_id').size().reset_index(name='order_size')

# 2. Define the categorisation function (1-5 Small, 6-15 Medium, 16+ Large)
def get_size_cat(size):
    if size <= 5: return 'Small'
    elif size <= 15: return 'Medium'
    else: return 'Large'

basket_sizes['size_cat'] = basket_sizes['order_size'].apply(get_size_cat)

# 3. Merge with the ORDERS table to get the 'order_hour_of_day' column
size_vs_hour = basket_sizes.merge(orders[['order_id', 'order_hour_of_day']], on='order_id')

# 4. Create the Contingency Table (Hour vs Size Category)
size_hour_ct = pd.crosstab(size_vs_hour['order_hour_of_day'], size_vs_hour['size_cat'])

print("Cross-tabulation: Order Hour vs. Order Size Category")
print(size_hour_ct)

Cross-tabulation: Order Hour vs. Order Size Category
size_cat           Large  Medium  Small
order_hour_of_day                      
0                   4217   10565   6590
1                   2186    5623   3787
2                   1274    3451   2345
3                    984    2484   1652
4                   1032    2489   1654
5                   1624    4312   3036
6                   5639   13887   9266
7                  17766   42156  26734
8                  34261   80522  53538
9                  48391  115714  79391
10                 54442  130757  86686
11                 54045  130089  83872
12                 51365  126148  78693
13                 52021  129016  80137
14                 51697  132448  81411
15                 50155  133760  82217
16                 46630  130266  79053
17                 37339  110046  66695
18                 28106   89070  53822
19                 21609   68389  41622
20                 18345   48923  30841
21                 16666   